# "rs-client-libraries" flow example with Prefect (not Dask)

See the associated:

  * Python module: [rs_client_flow.py](./rs_client_flow.py)
  * YAML file: [rs_client_flow.yaml](./rs_client_flow.yaml)

## Initialization

In [ ]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *

init_demo()

# Reload the global vars again
from resources.utils import *  

In [ ]:
# Other imports
import os
from importlib import reload
from rs_common import prefect_utils
from rs_common.prefect_utils import *
import resources

# Get the prefect share bucket folder
share_bucket = await get_share_bucket()

In [ ]:
# Set env vars for the prefect flow
os.environ["RS_SERVER_STAGING_ADDRESS"] = \
    os.environ["RSPY_WEBSITE"] if cluster_mode else \
    os.environ["RSPY_HOST_STAGING"]
os.environ["RS_API_KEY"] = "TO_BE_DEFINED" # should be passed as a prefect block ?
os.environ["RS_OWNER"] = OWNER_ID # jupyter username

## Deploy Prefect flow

We deploy our source code via the S3 bucket.

In [ ]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{OWNER_ID}/code" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{share_bucket.bucket_name}/{share_bucket.bucket_folder}/{s3_code_folder}'")

# Upload local directory and resources contents
await share_bucket.put_directory(local_path = ".", to_path = s3_code_folder)
await share_bucket.put_directory(local_path = resources.__path__[0], to_path = f"{s3_code_folder}/resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = f"{share_bucket.bucket_folder}/{s3_code_folder}"

In [ ]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./rs_client_flow.yaml"

In [ ]:
deploy_name = "get-staging-jobs/sprint19-rs-client"
await prefect_utils.wait_for_deployment(deploy_name)

## Run Prefect flow

In [ ]:
%%bash -s "$deploy_name"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --watch

NOTE: we could also call the Prefect flow from Python code. This is useful to debug.

In [ ]:
run_from_python = False
if run_from_python:
    # Import the module, or reload it if you changed its source code
    import rs_client_flow
    reload(rs_client_flow)
    
    # Run the flow
    results = rs_client_flow.get_staging_jobs()
    display(results)